# Pixels to Predictions — SmolVLM-500M Starter

## 0. Mount Google Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')
!ls /content/drive/MyDrive/data

Mounted at /content/drive
images.zip  sample_submission.csv  test.csv  train.csv	val.csv


## 1. Install packages

In [2]:
!pip install -q -U transformers==4.49.0 peft==0.13.2 accelerate==1.0.1
!pip install -q pillow pandas tqdm num2words safetensors

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 66.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 320.7/320.7 kB 25.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 330.9/330.9 kB 19.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 35.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 73.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.5/163.5 kB 8.8 MB/s eta 0:00:00


In [ ]:
import transformers, peft
print("transformers:", transformers.__version__)
print("peft:", peft.__version__)

import importlib.util
print("bitsandbytes installed:", importlib.util.find_spec("bitsandbytes") is not None)

from transformers.models.auto.configuration_auto import CONFIG_MAPPING
print("idefics3 in CONFIG_MAPPING:", "idefics3" in CONFIG_MAPPING)

transformers: 4.49.0
peft: 0.13.2
bitsandbytes installed: False
idefics3 in CONFIG_MAPPING: True


## 2. Imports, seeds, paths

In [4]:
import os, json, random, shutil, zipfile, time
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from tqdm.auto import tqdm

from transformers import AutoProcessor, AutoModelForVision2Seq, get_cosine_schedule_with_warmup
from peft import LoraConfig, get_peft_model, TaskType

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE, "| GPU:", torch.cuda.get_device_name(0) if DEVICE=="cuda" else "-")

Device: cuda | GPU: Tesla T4


In [ ]:
VERSION = "v3"   # change this in each copy: "v1", "v2", "v3", ...

# Drive locations (shared across versions)
DRIVE_ROOT = Path("/content/drive/MyDrive")
DATA_DRIVE = DRIVE_ROOT / "data"          # CSVs + images.zip live here
IMG_ZIP    = DATA_DRIVE / "images.zip"

# Local (ephemeral, per-session) cache for fast image I/O
LOCAL_ROOT     = Path("/content")
LOCAL_IMG_ZIP  = LOCAL_ROOT / "images.zip"
LOCAL_IMG_ROOT = LOCAL_ROOT                  # extract here -> /content/images/{train,val,test}/
SENTINEL       = LOCAL_ROOT / ".images_extracted"

# Output folder on Drive
OUT_DIR = DRIVE_ROOT / VERSION / "outputs"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# required files present on Drive
for p in [DATA_DRIVE/"train.csv", DATA_DRIVE/"val.csv", DATA_DRIVE/"test.csv",
          DATA_DRIVE/"sample_submission.csv", IMG_ZIP]:
    assert p.exists(), f"Missing on Drive: {p}"

CONFIG = dict(
    model_id            = "HuggingFaceTB/SmolVLM-500M-Instruct",
    image_longest_edge  = 384,          # unchanged — image res isn't the bottleneck
    lora_r              = 24,           # was 16 — biggest single change
    lora_alpha          = 48,           # was 32 — keep alpha/r ratio at 2
    lora_dropout        = 0.05,         # back to 0.05; with more capacity, 0.10 over-regularizes
    epochs              = 3,            # back to 3 — v2's epoch 4 contributed nothing
    lr                  = 1e-4,         # back to v1's LR; the slow LR bought nothing
    batch_size          = 2,
    grad_accum          = 8,
    warmup_ratio        = 0.03,
    max_text_tokens     = 1280,         # ↑ from 1024 — solution text adds length
    use_cot             = True,         # keep
    cot_prob            = 0.5,
    val_every_steps     = 50,           # keep frequent evals
    save_path           = str(OUT_DIR / "lora_adapter"),
    ckpt_latest_path    = str(OUT_DIR / "ckpt_latest"),
    ckpt_every_steps    = 50,
)
print(f"VERSION = {VERSION}")
print(f"Outputs: {OUT_DIR}")

VERSION = v3
Outputs: /content/drive/MyDrive/v3/outputs


## 3. Copy + extract `images.zip` to local (cached)

Reading thousands of small PNGs straight from Drive is ~100× slower than local disk — every file is a separate network call. This cell copies `images.zip` from Drive to `/content/` and extracts once. A sentinel `/content/.images_extracted` makes re-runs within the same Colab session a no-op. When the VM recycles (~12h or on disconnect), this runs again from scratch (2–5 min).

In [6]:
def ensure_local_images():
    if SENTINEL.exists():
        print(f"Images already extracted at {LOCAL_IMG_ROOT/'images'} — skipping.")
        return
    if not LOCAL_IMG_ZIP.exists():
        size_mb = IMG_ZIP.stat().st_size / 1e6
        print(f"Copying {IMG_ZIP.name} ({size_mb:.0f} MB) from Drive -> /content/ ...")
        t0 = time.time()
        shutil.copy(IMG_ZIP, LOCAL_IMG_ZIP)
        print(f"  copy done in {time.time()-t0:.1f}s")
    print("Extracting ...")
    t0 = time.time()
    with zipfile.ZipFile(LOCAL_IMG_ZIP) as z:
        z.extractall(LOCAL_IMG_ROOT)
    print(f"  extract done in {time.time()-t0:.1f}s")
    for split in ("train", "val", "test"):
        d = LOCAL_IMG_ROOT / "images" / split
        assert d.exists(), (
            f"Expected {d} after extract. Did you zip with `images/` as the top-level dir? "
            f"From the parent of `images/`, run: zip -r images.zip images/"
        )
        n = sum(1 for _ in d.glob("*.png"))
        print(f"  images/{split}: {n} PNGs")
    SENTINEL.touch()
    LOCAL_IMG_ZIP.unlink(missing_ok=True)   # free disk; sentinel protects against re-extract

ensure_local_images()

Copying images.zip (375 MB) from Drive -> /content/ ...
  copy done in 9.6s
Extracting ...
  extract done in 4.4s
  images/train: 3109 PNGs
  images/val: 1048 PNGs
  images/test: 1008 PNGs


## 4. Load CSVs (from Drive) and wire image paths to local extraction

In [7]:
def load_split(name):
    df = pd.read_csv(DATA_DRIVE / f"{name}.csv")
    df["choices"] = df["choices"].apply(json.loads)
    # CSV paths look like "images/train/train_00001.png";
    # after extraction they resolve against LOCAL_IMG_ROOT = /content/
    df["image_abs"] = df["image_path"].apply(lambda p: str(LOCAL_IMG_ROOT / p))
    return df

train_df = load_split("train")
val_df   = load_split("val")
test_df  = load_split("test")

print(f"train={len(train_df)}  val={len(val_df)}  test={len(test_df)}")
print("num_choices dist (train):", train_df.num_choices.value_counts().sort_index().to_dict())
print("answer dist   (train):",   train_df.answer.value_counts().sort_index().to_dict())

missing = [p for p in train_df.image_abs.sample(50, random_state=0) if not Path(p).exists()]
assert not missing, f"missing image samples: {missing[:3]}"
print("Image path sanity: OK")

train=3109  val=1048  test=1008
num_choices dist (train): {2: 664, 3: 1552, 4: 783, 5: 110}
answer dist   (train): {0: 1124, 1: 1028, 2: 737, 3: 204, 4: 16}
Image path sanity: OK


## 5. Load SmolVLM + attach LoRA (language-model only)

In [8]:
processor = AutoProcessor.from_pretrained(CONFIG["model_id"])
if hasattr(processor, "image_processor"):
    processor.image_processor.size = {"longest_edge": CONFIG["image_longest_edge"]}

model = AutoModelForVision2Seq.from_pretrained(
    CONFIG["model_id"], torch_dtype=torch.float16, low_cpu_mem_usage=True,
).to(DEVICE)

for p in model.parameters():
    p.requires_grad = False

target_modules = []
for name, _ in model.named_modules():
    if any(name.endswith(f".{proj}") for proj in ("q_proj","k_proj","v_proj","o_proj")):
        if any(tag in name for tag in ("text_model","language_model","text_decoder")):
            target_modules.append(name)

if not target_modules:
    for name, _ in model.named_modules():
        if any(name.endswith(f".{proj}") for proj in ("q_proj","k_proj","v_proj","o_proj")):
            if "vision" not in name and "visual" not in name:
                target_modules.append(name)

assert target_modules, "No LoRA target modules found — inspect model.named_modules()"
print(f"LoRA adapting {len(target_modules)} modules. Samples:")
for n in target_modules[:4]: print(" ", n)

lora_cfg = LoraConfig(
    r=CONFIG["lora_r"], lora_alpha=CONFIG["lora_alpha"], lora_dropout=CONFIG["lora_dropout"],
    bias="none", task_type=TaskType.CAUSAL_LM, target_modules=target_modules,
)
model = get_peft_model(model, lora_cfg)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"Trainable: {trainable:,}  ({trainable/total:.2%} of {total:,})")
assert trainable <= 5_000_000, f"OVER 5M CAP: {trainable:,}"
print("Under 5M cap ✓")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


processor_config.json:   0%|          | 0.00/68.0 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/429 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/486 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

special_tokens_map.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.02G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/136 [00:00<?, ?B/s]

LoRA adapting 128 modules. Samples:
  model.text_model.layers.0.self_attn.q_proj
  model.text_model.layers.0.self_attn.k_proj
  model.text_model.layers.0.self_attn.v_proj
  model.text_model.layers.0.self_attn.o_proj
Trainable: 4,915,200  (0.96% of 512,397,504)
Under 5M cap ✓


## 6. Prompt + letter-token scoring

In [9]:
LETTERS = ["A", "B", "C", "D", "E"]

def resolve_letter_token_ids(tokenizer, letters=LETTERS):
    ids = []
    for L in letters:
        tok_id = None
        for candidate in (f" {L}", L, f"\n{L}"):
            enc = tokenizer.encode(candidate, add_special_tokens=False)
            if len(enc) == 1:
                tok_id = enc[0]; break
        if tok_id is None:
            tok_id = tokenizer.encode(f" {L}", add_special_tokens=False)[-1]
        ids.append(tok_id)
    return ids

LETTER_IDS = resolve_letter_token_ids(processor.tokenizer)
print("Letter token IDs:", dict(zip(LETTERS, LETTER_IDS)))

def build_user_text(row, include_solution=False):
    parts = []
    if isinstance(row.get("lecture"), str) and row["lecture"].strip():
        parts.append(f"Lecture: {row['lecture'].strip()}")
    if isinstance(row.get("hint"), str) and row["hint"].strip():
        parts.append(f"Context: {row['hint'].strip()}")
    parts.append(f"Question: {row['question'].strip()}")
    choice_lines = "\n".join(f"{LETTERS[i]}. {c}" for i, c in enumerate(row["choices"]))
    parts.append("Choices:\n" + choice_lines)
    if include_solution and isinstance(row.get("solution"), str) and row["solution"].strip():
        parts.append(f"Reasoning hint: {row['solution'].strip()}")
    valid = "".join(LETTERS[: int(row["num_choices"]) ])
    parts.append(f"Respond with a single letter from [{', '.join(valid)}].")
    parts.append("Answer:")
    return "\n\n".join(parts)

def build_prompt(row, include_solution=False):
    messages = [{"role":"user", "content":[
        {"type":"image"},
        {"type":"text","text": build_user_text(row, include_solution=include_solution)},
    ]}]
    return processor.apply_chat_template(messages, add_generation_prompt=True)

print(build_prompt(train_df.iloc[0].to_dict())[:800], "...")

Letter token IDs: {'A': 330, 'B': 389, 'C': 340, 'D': 422, 'E': 414}
<|im_start|>User:<image>Lecture: Animals increase their reproductive success when they have offspring that survive to reproduce.
Animals can increase their chances of having offspring by behaving in ways that help them get partners to mate and reproduce with. These partners are called mates. For example, animals may make special sounds, perform specific dances, or show off bright colors to attract mates. Animals may also compete with each other for mates.
Animals can increase the chances that their offspring will survive to reproduce by caring for and protecting them. For example, animals may feed their offspring or guard them from predators. These behaviors increase the chances that the offspring will survive to adulthood, when they can reproduce.
Many behaviors can increase the chances t ...


## 7. Dataset / collators

In [10]:
class VQADataset(Dataset):
    def __init__(self, df, mode, cot_prob=0.0):
        self.df = df.reset_index(drop=True)
        self.mode = mode
        self.cot_prob = cot_prob
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx].to_dict()
        img = Image.open(row["image_abs"]).convert("RGB")
        include_sol = (self.mode=="train" and self.cot_prob>0
                       and isinstance(row.get("solution"), str)
                       and random.random() < self.cot_prob)
        prompt = build_prompt(row, include_solution=include_sol)
        item = dict(image=img, prompt=prompt, num_choices=int(row["num_choices"]), id=row["id"])
        if self.mode=="train": item["answer"] = int(row["answer"])
        return item

def collate_train(batch):
    images  = [b["image"] for b in batch]
    prompts = [b["prompt"] for b in batch]
    answers = [b["answer"] for b in batch]
    full_texts = [p + " " + LETTERS[a] for p, a in zip(prompts, answers)]
    enc = processor(text=full_texts, images=images, return_tensors="pt",
                    padding=True, truncation=True, max_length=CONFIG["max_text_tokens"])
    input_ids = enc["input_ids"]
    labels = input_ids.clone(); labels[:] = -100
    attn = enc["attention_mask"]
    last_idx = attn.sum(dim=1) - 1
    for i, pos in enumerate(last_idx.tolist()):
        labels[i, pos] = input_ids[i, pos]
    enc["labels"] = labels
    return enc

def collate_infer(batch):
    images  = [b["image"] for b in batch]
    prompts = [b["prompt"] for b in batch]
    num_ch  = torch.tensor([b["num_choices"] for b in batch], dtype=torch.long)
    ids     = [b["id"] for b in batch]
    enc = processor(text=prompts, images=images, return_tensors="pt",
                    padding=True, truncation=True, max_length=CONFIG["max_text_tokens"])
    enc["num_choices"] = num_ch
    enc["ids"] = ids
    return enc

## 8. Inference / eval helpers

In [11]:
@torch.no_grad()
def predict(model, df, batch_size=4):
    model.eval()
    ds = VQADataset(df, mode="infer")
    dl = DataLoader(ds, batch_size=batch_size, shuffle=False, collate_fn=collate_infer, num_workers=2)
    all_preds, all_ids = [], []
    letter_ids_tensor = torch.tensor(LETTER_IDS, device=DEVICE)
    NEG = torch.finfo(torch.float16).min

    for enc in tqdm(dl, desc="predict"):
        num_ch = enc.pop("num_choices")
        ids    = enc.pop("ids")
        enc = {k: v.to(DEVICE) for k, v in enc.items()}
        with torch.amp.autocast("cuda", dtype=torch.float16):
            out = model(**enc)
        logits = out.logits
        last_idx = enc["attention_mask"].sum(dim=1) - 1
        gathered = logits[torch.arange(logits.size(0)), last_idx]
        letter_logits = gathered[:, letter_ids_tensor]
        mask = torch.arange(5, device=DEVICE).unsqueeze(0) >= num_ch.to(DEVICE).unsqueeze(1)
        letter_logits = letter_logits.masked_fill(mask, NEG)
        preds = letter_logits.argmax(dim=-1).cpu().tolist()
        all_preds.extend(preds); all_ids.extend(ids)
    model.train()
    return pd.DataFrame({"id": all_ids, "answer": all_preds})

def val_accuracy(model):
    preds = predict(model, val_df)
    merged = preds.merge(val_df[["id","answer"]], on="id", suffixes=("_pred","_gold"))
    return (merged.answer_pred == merged.answer_gold).mean(), preds

## 9. Visualization helpers

In [12]:
class TrainingLogger:
    """Persistent training log. Reloads from JSON on init so it survives Colab disconnects."""
    def __init__(self, log_path):
        self.path = Path(log_path)
        self.path.parent.mkdir(parents=True, exist_ok=True)
        self.data = {
            "step_metrics": [],
            "eval_metrics": [],
            "events": [],
            "meta": {
                "start_time": time.time(),
                "config": None,
                "version": None,
                "zero_shot_val_acc": None,
                "total_train_seconds": 0.0,
                "peak_gpu_mem_mb": 0.0,
                "trainable_params": None,
                "finished": False,
            },
            "final": {
                "val_breakdown": None,
                "test_pred_distribution": None,
                "best_val_acc": None,
                "best_val_step": None,
            },
        }
        if self.path.exists():
            try:
                with open(self.path) as f:
                    self.data = json.load(f)
                print(f"Loaded existing log ({len(self.data['step_metrics'])} steps, "
                      f"{len(self.data['eval_metrics'])} evals)")
            except Exception as e:
                print(f"Could not load log at {self.path}: {e}. Starting fresh.")

    def log_step(self, step, epoch, loss, lr, grad_norm, time_s, gpu_mem_mb):
        self.data["step_metrics"].append({
            "step": step, "epoch": epoch,
            "loss": float(loss), "lr": float(lr),
            "grad_norm": float(grad_norm), "time_s": float(time_s),
            "gpu_mem_mb": float(gpu_mem_mb),
        })
        self.data["meta"]["peak_gpu_mem_mb"] = max(
            self.data["meta"]["peak_gpu_mem_mb"], float(gpu_mem_mb))

    def log_eval(self, step, epoch, val_acc, breakdown=None):
        entry = {"step": step, "epoch": epoch, "val_acc": float(val_acc)}
        if breakdown is not None:
            entry["breakdown"] = breakdown
        self.data["eval_metrics"].append(entry)

    def log_event(self, step, event, detail=""):
        self.data["events"].append({
            "step": step, "event": event, "detail": detail, "time": time.time()})

    def update_meta(self, **kwargs):
        self.data["meta"].update(kwargs)

    def update_final(self, **kwargs):
        self.data["final"].update(kwargs)

    def save(self):
        tmp = self.path.with_suffix(".tmp.json")
        with open(tmp, "w") as f:
            json.dump(self.data, f, indent=2, default=str)
        tmp.replace(self.path)


def breakdown_metrics(val_df_with_gold, preds_df):
    """Slice val accuracy by num_choices / subject / grade / gold answer index."""
    m = preds_df.rename(columns={"answer": "pred"}).merge(
        val_df_with_gold[["id","answer","num_choices","subject","grade"]], on="id")
    m["correct"] = (m.pred == m.answer).astype(int)

    def by_col(col):
        g = m.groupby(col).agg(n=("correct","size"), acc=("correct","mean"))
        return {str(k): {"n": int(v["n"]), "acc": float(v["acc"])} for k, v in g.to_dict("index").items()}

    conf = {}
    for (gold, pred), cnt in m.groupby(["answer","pred"]).size().items():
        conf.setdefault(str(gold), {})[str(pred)] = int(cnt)

    return {
        "overall_acc": float(m.correct.mean()),
        "by_num_choices": by_col("num_choices"),
        "by_subject":     by_col("subject"),
        "by_grade":       by_col("grade"),
        "by_answer_idx":  by_col("answer"),
        "pred_distribution": {str(k): int(v) for k, v in m.pred.value_counts().sort_index().items()},
        "gold_distribution": {str(k): int(v) for k, v in m.answer.value_counts().sort_index().items()},
        "confusion": conf,
    }


LOG_PATH = OUT_DIR / "metrics.json"
logger = TrainingLogger(LOG_PATH)
logger.update_meta(config=CONFIG, version=VERSION)
logger.save()
print(f"Logger writing to {LOG_PATH}")

Logger writing to /content/drive/MyDrive/v3/outputs/metrics.json


## 10. Zero-shot baseline

In [13]:
acc0, _ = val_accuracy(model)
print(f"Zero-shot val accuracy: {acc0:.4f}  (floor to beat: 0.36 = predict-all-zeros)")
logger.update_meta(zero_shot_val_acc=float(acc0))
logger.log_event(0, "zero_shot", f"acc={acc0:.4f}")
logger.save()

predict:   0%|          | 0/262 [00:00<?, ?it/s]

Zero-shot val accuracy: 0.5076  (floor to beat: 0.36 = predict-all-zeros)


## 11. Checkpoint Saving

In [14]:
from safetensors.torch import load_file as safe_load
from peft.utils import set_peft_model_state_dict   # <-- the missing import

def save_full_ckpt(path, model, optim, sched, scaler, epoch, batches_in_epoch, best_val):
    """Save LoRA adapter + optimizer + scheduler + scaler + counters + RNG."""
    p = Path(path); p.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(str(p))
    torch.save({
        "optim":  optim.state_dict(),
        "sched":  sched.state_dict(),
        "scaler": scaler.state_dict(),
        "epoch":  epoch,
        "batches_in_epoch": batches_in_epoch,
        "best_val": best_val,
        "rng_torch":  torch.get_rng_state(),
        "rng_cuda":   torch.cuda.get_rng_state_all(),
        "rng_numpy":  np.random.get_state(),
        "rng_python": random.getstate(),
    }, p / "trainer_state.pt")

def _lora_fingerprint(model):
    """Frobenius norm of the first LoRA A matrix — a quick fingerprint for save/load consistency."""
    for n, p in model.named_parameters():
        if "lora_A" in n and p.requires_grad:
            return float(p.detach().float().norm().item())
    return float("nan")

def try_load_full_ckpt(path, model, optim, sched, scaler):
    """Returns (epoch, batches_in_epoch, best_val). (0, 0, -1.0) if no ckpt."""
    p = Path(path)
    state_file   = p / "trainer_state.pt"
    adapter_file = p / "adapter_model.safetensors"
    if not state_file.exists() or not adapter_file.exists():
        print(f"No checkpoint at {p} — starting fresh.")
        return 0, 0, -1.0

    print(f"Resuming from {p}")
    before = _lora_fingerprint(model)

    # PEFT-aware load: translates save-format keys → live-model keys (adapter name insertion)
    adapter_sd = safe_load(str(adapter_file))
    result = set_peft_model_state_dict(model, adapter_sd)
    # result.unexpected_keys should be empty for a matching adapter
    unexpected = getattr(result, "unexpected_keys", [])
    if unexpected:
        raise RuntimeError(f"unexpected keys after load: {unexpected[:3]}")

    after = _lora_fingerprint(model)
    print(f"  lora_A fingerprint: {before:.6f} -> {after:.6f}"
          + ("  (changed ✓)" if before != after else "  (UNCHANGED — load did nothing!)"))

    state = torch.load(state_file, map_location="cpu", weights_only=False)
    optim.load_state_dict(state["optim"])
    sched.load_state_dict(state["sched"])
    scaler.load_state_dict(state["scaler"])
    torch.set_rng_state(state["rng_torch"])
    torch.cuda.set_rng_state_all(state["rng_cuda"])
    np.random.set_state(state["rng_numpy"])
    random.setstate(state["rng_python"])
    print(f"  resumed at epoch={state['epoch']}, batches_in_epoch={state['batches_in_epoch']}, "
          f"best_val={state['best_val']:.4f}")
    return state["epoch"], state["batches_in_epoch"], state["best_val"]

## 12. Train LoRA

In [15]:
from itertools import islice

def train(model, cfg, logger):
    train_ds = VQADataset(train_df, mode="train",
                          cot_prob=cfg["cot_prob"] if cfg["use_cot"] else 0.0)
    n_batches = len(train_ds) // cfg["batch_size"]
    steps_per_epoch = n_batches // cfg["grad_accum"]
    total_steps  = steps_per_epoch * cfg["epochs"]
    warmup_steps = int(total_steps * cfg["warmup_ratio"])

    optim  = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad],
                               lr=cfg["lr"], weight_decay=0.0)
    sched  = get_cosine_schedule_with_warmup(optim, warmup_steps, total_steps)
    scaler = torch.amp.GradScaler("cuda")   # <-- new API

    start_epoch, start_batches, best_val = try_load_full_ckpt(
        cfg["ckpt_latest_path"], model, optim, sched, scaler)
    if start_epoch > 0 or start_batches > 0:
        logger.log_event(len(logger.data["step_metrics"]),
                         "resumed", f"epoch={start_epoch}, batches={start_batches}")

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    logger.update_meta(trainable_params=int(trainable))

    model.train()
    global_step = sched.state_dict().get("_step_count", 0)
    run_start   = time.time()
    batch_timer = time.time()
    running     = 0.0

    for epoch in range(start_epoch, cfg["epochs"]):
        g = torch.Generator(); g.manual_seed(SEED + epoch)
        dl = DataLoader(train_ds, batch_size=cfg["batch_size"], shuffle=True,
                        collate_fn=collate_train, num_workers=2, drop_last=True, generator=g)
        skip_n = start_batches if epoch == start_epoch else 0
        if skip_n: print(f"  skipping {skip_n} batches to resume")

        pbar = tqdm(islice(dl, skip_n, None),
                    desc=f"epoch {epoch+1}/{cfg['epochs']}",
                    initial=skip_n, total=len(dl))
        optim.zero_grad()
        batches_seen = skip_n

        for enc in pbar:
            batches_seen += 1
            enc = {k: v.to(DEVICE) for k, v in enc.items()}
            with torch.amp.autocast("cuda", dtype=torch.float16):   # <-- new API
                out  = model(**enc)
                loss = out.loss / cfg["grad_accum"]
            scaler.scale(loss).backward()
            running += loss.item() * cfg["grad_accum"]

            if batches_seen % cfg["grad_accum"] == 0:
                scaler.unscale_(optim)
                grad_norm = torch.nn.utils.clip_grad_norm_(
                    [p for p in model.parameters() if p.requires_grad], 1.0)
                scaler.step(optim); scaler.update(); sched.step(); optim.zero_grad()
                global_step += 1
                step_time = time.time() - batch_timer
                batch_timer = time.time()
                mem_mb = torch.cuda.max_memory_allocated() / 1e6

                step_loss = running / cfg["grad_accum"]
                logger.log_step(global_step, epoch,
                                loss=step_loss, lr=sched.get_last_lr()[0],
                                grad_norm=grad_norm.item(), time_s=step_time,
                                gpu_mem_mb=mem_mb)
                running = 0.0
                pbar.set_postfix(loss=f"{step_loss:.3f}",
                                 lr=f"{sched.get_last_lr()[0]:.2e}",
                                 gn=f"{grad_norm.item():.2f}")

                if global_step % cfg["ckpt_every_steps"] == 0:
                    save_full_ckpt(cfg["ckpt_latest_path"], model, optim, sched, scaler,
                                   epoch, batches_seen, best_val)
                    logger.save()

                if global_step % cfg["val_every_steps"] == 0:
                    acc, preds = val_accuracy(model)
                    bd = breakdown_metrics(val_df, preds)
                    logger.log_eval(global_step, epoch, acc, breakdown=bd)
                    print(f"  step {global_step}: val={acc:.4f}")
                    if acc > best_val:
                        best_val = acc
                        model.save_pretrained(cfg["save_path"])
                        logger.log_event(global_step, "best_adapter_saved", f"val={acc:.4f}")
                        print(f"  ↑ saved best adapter (val={acc:.4f})")
                    logger.save()
                    model.train()

        acc, preds = val_accuracy(model)
        bd = breakdown_metrics(val_df, preds)
        logger.log_eval(global_step, epoch, acc, breakdown=bd)
        print(f"epoch {epoch+1} end: val={acc:.4f}")
        if acc > best_val:
            best_val = acc
            model.save_pretrained(cfg["save_path"])
            logger.log_event(global_step, "best_adapter_saved", f"val={acc:.4f}")
        save_full_ckpt(cfg["ckpt_latest_path"], model, optim, sched, scaler,
                       epoch + 1, 0, best_val)
        logger.update_meta(total_train_seconds=
            logger.data["meta"]["total_train_seconds"] + (time.time() - run_start))
        logger.save()
        run_start = time.time()
        model.train()
        start_batches = 0

    logger.update_meta(finished=True)
    logger.save()
    return best_val

best_val = train(model, CONFIG, logger)
print(f"Best val accuracy: {best_val:.4f}")

No checkpoint at /content/drive/MyDrive/v3/outputs/ckpt_latest — starting fresh.


epoch 1/3:   0%|          | 0/1554 [00:00<?, ?it/s]

/tmp/ipykernel_4589/282264891.py:57: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scaler.step(optim); scaler.update(); sched.step(); optim.zero_grad()


predict:   0%|          | 0/262 [00:00<?, ?it/s]

  step 50: val=0.6985
  ↑ saved best adapter (val=0.6985)


...

  step 500: val=0.7672
  ↑ saved best adapter (val=0.7672)


predict:   0%|          | 0/262 [00:00<?, ?it/s]

  step 550: val=0.7719
  ↑ saved best adapter (val=0.7719)


predict:   0%|          | 0/262 [00:00<?, ?it/s]

epoch 3 end: val=0.7729
Best val accuracy: 0.7729


## 13. Final val eval + test inference + submission.csv

In [16]:
val_acc, _ = val_accuracy(model)
print(f"Final val accuracy: {val_acc:.4f}")

predict:   0%|          | 0/262 [00:00<?, ?it/s]

Final val accuracy: 0.7729


In [17]:
test_preds = predict(model, test_df, batch_size=4)

sample = pd.read_csv(DATA_DRIVE / "sample_submission.csv")
assert set(test_preds.columns) == {"id","answer"}, test_preds.columns
assert set(test_preds.id) == set(sample.id), "ID set does not match sample_submission"
test_preds = test_preds.set_index("id").loc[sample.id].reset_index()
test_preds["answer"] = test_preds["answer"].astype(int)

check = test_preds.merge(test_df[["id","num_choices"]], on="id")
assert (check.answer < check.num_choices).all(), "some predictions exceed num_choices"

sub_path = OUT_DIR / "submission.csv"
test_preds.to_csv(sub_path, index=False)
print(f"Wrote {sub_path}  ({len(test_preds)} rows)")

# Final eval + logging for analyze.py to consume
final_acc, final_preds = val_accuracy(model)
final_bd = breakdown_metrics(val_df, final_preds)

evals = logger.data["eval_metrics"]
best_eval = max(evals, key=lambda e: e["val_acc"]) if evals else None

logger.update_final(
    val_breakdown=final_bd,
    test_pred_distribution={str(k): int(v) for k, v in test_preds.answer.value_counts().sort_index().items()},
    best_val_acc=best_eval["val_acc"] if best_eval else None,
    best_val_step=best_eval["step"]   if best_eval else None,
    final_val_acc=float(final_acc),
)
logger.save()

print(f"Final val acc: {final_acc:.4f}")
print("Answer distribution:", test_preds.answer.value_counts().sort_index().to_dict())
print(f"metrics.json at {logger.path}")

predict:   0%|          | 0/252 [00:00<?, ?it/s]

Wrote /content/drive/MyDrive/v3/outputs/submission.csv  (1008 rows)


predict:   0%|          | 0/262 [00:00<?, ?it/s]

Final val acc: 0.7729
Answer distribution: {0: 371, 1: 338, 2: 229, 3: 69, 4: 1}
metrics.json at /content/drive/MyDrive/v3/outputs/metrics.json


## Notes

- **Per-version workflow**: duplicate the notebook in `Colab Notebooks/`, change `VERSION` at the top, and run. Adapter + `submission.csv` go to `MyDrive/{VERSION}/outputs/`, isolated from other versions.
- **Image cache lifecycle**: `ensure_local_images()` is a no-op after the first run within a Colab session. When the VM recycles (disconnect or ~12h), the sentinel and extracted images disappear and it runs again. CSV reads always come from Drive — they're small.
- **Kaggle offline submission**: Kaggle's final eval has no internet. To port: upload base SmolVLM weights as a Kaggle Dataset, upload your best LoRA adapter as another Dataset, and change `model_id` / adapter load path to those local Kaggle paths.
- **If Colab OOMs**: drop `image_longest_edge` to 336, set `batch_size=1` and `grad_accum=16`, or cut `max_text_tokens` to 768.
- **Ideas for later versions**: weighted sampling on the answer-index imbalance; ensemble plain + CoT adapters by averaging letter logits before masked argmax; swap LoRA targets to just `q_proj, v_proj` and push rank to 32 within the 5M cap.